In [ ]:
using Pkg
Pkg.activate("./")
Pkg.develop(path="../../")
Pkg.develop(path="../../../TauRunner/TauRunner.jl/")
ENV["TAMBOSIM_PATH"] = realpath("../../")

In [ ]:
using Tambo
using Makie
using CairoMakie
using HDF5
using LinearAlgebra
using Unitful
using StatsBase
include("../plotting_boilerplate.jl")

In [ ]:
filename = "./triangulation.h5"
key = "colca_valley_30000"

earth = Tambo.Earth("$(filename):$(key)")
enu_coordinates = Tambo.CoordinateSystem(earth)

In [ ]:
# centroids = Tambo.centroid.(earth.topography)
# xys = [[c[1], c[2]] for c in centroids]

coords = Tambo.Coordinate[]
for line in readlines("/Users/jlazar/Downloads/Default Dataset.csv")
    x, y = parse.(Float64, split(line, ", "))
    idx = argmin([sum(([x*u"m", y*u"m"] - xy) .^2) for xy in xys])
    coord = Tambo.Coordinate([x*u"m", y*u"m", centroids[idx][3]], enu_coordinates)
    push!(coords, coord)
end
    
# # This need to be oriented anti-clockwise
# cog = [sum([c[1] for c in coords]), sum([c[2] for c in coords])] ./ length(coords)
# phis = []
# for coord in coords
#     push!(phis, atan(coord[2]-cog[2], coord[1]-cog[1]))
# end
# # sorter = sortperm(phis)
# # coords = coords[sorter]

In [ ]:
# coords = Tambo.Coordinate[]
# for longlat in outline
#     x, y, z = Tambo.longlat_to_cart(longlat...) .* earth.prem[end].radius
#     c = Tambo.Coordinate(x, y, z, Tambo.ecefcoordinates)
#     c = convert(enu_coordinates, c)
#     push!(coords, c)
# end
# coords = coords[1:end-5]

In [ ]:
radius = 6 * u"km"
fig = Figure()

triangles = filter(triangle->norm(Tambo.centroid(triangle)[1:2]) < radius, earth.topography)
triangles = filter(triangle->Tambo.centroid(triangle)[3] > 0 * u"km", triangles)

vxs, faces = Tambo.triangles_to_mesh(triangles)

zaspect = 4500*Tambo.uparse("m") / (2 * radius)

ax = Axis3(
    fig[1, 1],
    azimuth=deg2rad(-90),
    elevation=deg2rad(90),
    aspect=(1, 1, zaspect),
    zgridvisible=false,
    zlabelvisible=false,
    zticklabelsvisible=false,
    zticksvisible=false,
    xlabel="x [m]",
    ylabel="y [m]",
)
zlims!(ax, 1000, 5500)

m = mesh!(
    ax,
    map(vx->Point3f(ustrip.(vx.point)), vxs),
    faces,
    color=[v.point.z.val for v in vxs],
    colormap=Reverse(:speed),
    colorrange=[2000, 4000]
)

lines!(
    ax,
    ustrip.(vcat([c[1] for c in coords], [coords[1][1]])),
    ustrip.(vcat([c[2] for c in coords], [coords[1][2]])),
    [2000.0 for _ in 1:length(coords)+1],
    linewidth=5,
    color=:crimson
)

# Axis 3: colorbar
colorbar = Colorbar(fig[1,2], m, label="Elevation [m]")

display(fig)

In [ ]:
function is_in_outline(
#     coords::Vector{Tambo.Coordinate}, 
#     test_point::Tuple{Float64,Float64},
    coords, 
    test_point
)
    outline_points = [(coord[1], coord[2]) for coord in coords]
    n = length(outline_points)
    inside = false

    j = n
    for i in 1:n
        xi, yi = outline_points[i]
        xj, yj = outline_points[j]
        px, py = test_point

        # Check if point is between the y-coordinates of the edge
        if ((yi > py) != (yj > py)) &&
           (px < (xj - xi) * (py - yi) / (yj - yi) + xi)
#                 inside = true
            inside = !inside
        end
        j = i
    end
    
    return inside
end

In [ ]:
zaspect = 4500*Tambo.uparse("m") / (2 * radius)
    
fig = Figure()
ax = Axis3(
    fig[1, 1],
    azimuth=deg2rad(-80),
    elevation=deg2rad(20),
    aspect=(1, 1, zaspect),
    zgridvisible=false,
    zlabelvisible=false,
    zticklabelsvisible=false,
    zticksvisible=false,
    xlabel="x [m]",
    ylabel="y [m]",
#         margins = (50, 50, 50, 50)
)
zlims!(ax, 1000, 5500)
    
triangles = filter(triangle->norm(Tambo.centroid(triangle)[1:2]) < radius, earth.topography)
triangles = filter(triangle->Tambo.centroid(triangle)[3] > 0 * u"km", triangles)

vxs, faces = Tambo.triangles_to_mesh(triangles)

m = mesh!(
    ax,
    map(vx->Point3f(ustrip.(vx.point)), vxs),
    faces,
    color=[v.point.z.val for v in vxs],
    colormap=Reverse(:speed),
    colorrange=[2000, 4000],
    alpha=0.1
)

test_points = []
for triangle in earth.topography
    c = Tambo.centroid(triangle)
    push!(test_points, (c[1], c[2]))
end

triangles = earth.topography[is_in_outline.(Ref(coords), test_points)]

vxs, faces = Tambo.triangles_to_mesh(triangles)

m = mesh!(
    ax,
    map(vx->Point3f(ustrip.(vx.point)), vxs),
    faces,
    color=[v.point.z.val for v in vxs],
    colormap=Reverse(:speed),
    colorrange=[2000, 4000]
) 

# Axis 3: colorbar
colorbar = Colorbar(fig[1,2], m, label="Elevation [m]")
    
fig

In [ ]:
findall(is_in_outline.(Ref(coords), test_points))

In [ ]:
h5open("../../resources/basic_geometry.h5", "r+") do h5f
    delete_object(h5f["colca_valley_30000/detector1"])
    h5f["colca_valley_30000/detector1"] = findall(is_in_outline.(Ref(coords), test_points))
end

In [ ]:
h5open("../../resources/basic_geometry.h5", "r+") do h5f

In [ ]:
radius = 6 * u"km"
fig = Figure()

triangles = filter(triangle->norm(Tambo.centroid(triangle)[1:2]) < radius, earth.topography)
triangles = filter(triangle->Tambo.centroid(triangle)[3] > 0 * u"km", triangles)

vxs, faces = Tambo.triangles_to_mesh(triangles)

zaspect = 4500*Tambo.uparse("m") / (2 * radius)

ax = Axis3(
    fig[1, 1],
    azimuth=deg2rad(-60),
    elevation=deg2rad(20),
    aspect=(1, 1, zaspect),
    zgridvisible=false,
    zlabelvisible=false,
    zticklabelsvisible=false,
    zticksvisible=false,
    xlabel="x [m]",
    ylabel="y [m]",
)
zlims!(ax, 1000, 5500)

m = mesh!(
    ax,
    map(vx->Point3f(ustrip.(vx.point)), vxs),
    faces,
    color=[v.point.z.val for v in vxs],
    colormap=Reverse(:speed),
    colorrange=[2000, 4000]
)

lines!(
    ax,
    ustrip.(vcat([c[1] for c in coords], [coords[1][1]])),
    ustrip.(vcat([c[2] for c in coords], [coords[1][2]])),
    ustrip.(vcat([c[3] for c in coords], [coords[1][3]])),
#     [2000.0 for _ in 1:length(coords)+1],
    linewidth=3,
    color=:crimson
)

# Axis 3: colorbar
colorbar = Colorbar(fig[1,2], m, label="Elevation [m]")

display(fig)

In [ ]:
radius = 6 * u"km"
fig = Figure(size=(1400, 700))

triangles = filter(triangle->norm(Tambo.centroid(triangle)[1:2]) < radius, earth.topography)
triangles = filter(triangle->Tambo.centroid(triangle)[3] > 0 * u"km", triangles)

vxs, faces = Tambo.triangles_to_mesh(triangles)

zaspect = 4500*Tambo.uparse("m") / (2 * radius)

ax = Axis3(
    fig[1, 1],
    azimuth=deg2rad(-90),
    elevation=deg2rad(90),
    aspect=(1, 1, zaspect),
    zgridvisible=false,
    zlabelvisible=false,
    zticklabelsvisible=false,
    zticksvisible=false,
    xlabel="x [m]",
    ylabel="y [m]",
)
zlims!(ax, 1000, 5500)

m = mesh!(
    ax,
    map(vx->Point3f(ustrip.(vx.point)), vxs),
    faces,
    color=[v.point.z.val for v in vxs],
    colormap=Reverse(:speed),
    colorrange=[2000, 4000]
)

lines!(
    ax,
    ustrip.(vcat([c[1] for c in coords], [coords[1][1]])),
    ustrip.(vcat([c[2] for c in coords], [coords[1][2]])),
    [2000.0 for _ in 1:length(coords)+1],
    linewidth=5,
    color=:crimson
)

ax = Axis3(
    fig[1, 2],
    azimuth=deg2rad(-60),
    elevation=deg2rad(20),
    aspect=(1, 1, zaspect),
    zgridvisible=false,
    zlabelvisible=false,
    zticklabelsvisible=false,
    zticksvisible=false,
    xlabel="x [m]",
    ylabel="y [m]",
)
zlims!(ax, 1000, 5500)

m = mesh!(
    ax,
    map(vx->Point3f(ustrip.(vx.point)), vxs),
    faces,
    color=[v.point.z.val for v in vxs],
    colormap=Reverse(:speed),
    colorrange=[2000, 4000]
)

lines!(
    ax,
    ustrip.(vcat([c[1] for c in coords], [coords[1][1]])),
    ustrip.(vcat([c[2] for c in coords], [coords[1][2]])),
    ustrip.(vcat([c[3] for c in coords], [coords[1][3]])),
#     [2000.0 for _ in 1:length(coords)+1],
    linewidth=3,
    color=:crimson
)

# Axis 3: colorbar
colorbar = Colorbar(fig[1,3], m, label="Elevation [m]")

colgap!(fig.layout, 50)


display(fig)